# Biodiversity Footprint Accounting

Computes the **consumption-based biodiversity footprint** (PDF, Potentially
Disappeared Fraction) for 2023 using the GLORIA MRIO table (164 countries x
120 sectors) and the Chaudhary et al. (2015) characterization factors (CF).


## Part 1 — Compute the consumption-based footprint (raw matrices)

In [3]:
import gc
import os
import time

import h5py
import numpy as np
import pandas as pd
import scipy.io

# ── Paths (adjust to your local data directory) ────────────────────────────
IO_DIR  = r'E:\phdstudy\IO'
HOU_DIR = r'E:\phdstudy\BiodiversityPHD-2ndpaper\household_expenditure'
CF_DIR  = r'E:\phdstudy\production\input\io_matrices'
OUT_DIR = r'E:\phdstudy\BiodiversityPHD-2ndpaper\submission_code\1_Biodiversity_footprint_accounting\output'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Constants ────────────────────────────────────────────────────────────
G, S, N = 164, 120, 201        # countries, sectors, income bins
R = G * S                      # 19,680 country-sector rows
EPSILON = 1e-9                 # numerical stabiliser for the Leontief inverse
AREA_TO_M2 = 1e7                # 1000 ha -> m^2
LU_NAMES = ['Annual_crops', 'Permanent_crops', 'Pasture',
            'Intensive_forestry', 'Extensive_forestry', 'Urban']
YEAR_COL = 33                   # 2023 column in the 1990-2029 SDG series

t0 = time.time()

# ── Step 1: characterization factors (CF), per sector ──────────────────────
print('[1/4] Building footprint intensity q ...')
cf = pd.read_csv(os.path.join(CF_DIR, 'GLORIA_Chaudhary_CF_long.csv'))
cf_country = np.zeros((G, 6))
for li, lu in enumerate(LU_NAMES):
    sub = cf[cf['LU_type'] == lu].sort_values('GLORIA_ID')
    cf_country[:, li] = sub['CF_median_PDF_per_m2'].fillna(0).values
cf_sector = np.repeat(cf_country, S, axis=0)          # (R, 6) broadcast to sector level
del cf; gc.collect()

# land use by sector (6 LU types, 2023), from the satellite account
mat_sdg = scipy.io.loadmat(os.path.join(IO_DIR, '99 SDG ind 1990-2029.mat'))
lu_2023 = mat_sdg['SDG'][27:33, :, YEAR_COL]           # (6, R), unit: 1000 ha
del mat_sdg; gc.collect()

# footprint intensity per sector: PDF = land use (m^2) x CF (PDF/m^2)
sat = np.einsum('ls,sl->s', lu_2023 * AREA_TO_M2, cf_sector)   # (R,)
del cf_sector, lu_2023; gc.collect()
print(f'    Total production-side footprint (2023): {sat.sum():.4e} PDF')

# ── Step 2: Leontief inverse and footprint multiplier ───────────────────────
print('[2/4] Building Leontief inverse L ... (5-20 min)')
with h5py.File(os.path.join(IO_DIR, 'UT2023.mat'), 'r') as hf:
    z_key = next(k for k in ['UT2023', 'Z', 'Z2023'] + list(hf.keys()) if k in hf)
    Z = hf[z_key][:].astype(np.float64)
with h5py.File(os.path.join(IO_DIR, 'FD2023.mat'), 'r') as hf:
    fd_key = next(k for k in ['FD', 'FD2023'] + list(hf.keys()) if k in hf)
    FD = hf[fd_key][:].T.astype(np.float64)

x = Z.sum(axis=1) + FD.sum(axis=1)                    # total sector output
x_safe = np.where(x > 0, x, 1.0)
del FD; gc.collect()

A = np.nan_to_num(Z / x_safe[np.newaxis, :], nan=0.0, posinf=0.0, neginf=0.0)
del Z; gc.collect()
L = np.linalg.inv(np.eye(R) * (1.0 + EPSILON) - A)
del A; gc.collect()

q = np.nan_to_num(sat / x_safe, nan=0.0, posinf=0.0)
q[x <= 0] = 0.0
del sat; gc.collect()

f_vec = q @ L                                          # total (direct + upstream) footprint multiplier, (R,)
print(f'    L done  [{time.time() - t0:.0f}s]')

# ── Step 3: consumption-side footprint by country x income bin ─────────────
print('[3/4] Applying household expenditure (Target) ...')
npz = np.load(os.path.join(HOU_DIR, 'Target_2023.npz'))
Target = np.nan_to_num(npz['Target'].astype(np.float64), nan=0.0)   # (R, G*N)
regnam = list(npz['regnam'])

fp_income = (f_vec @ Target).reshape(G, N)             # (164, 201) raw footprint matrix
expenditure = Target.sum(axis=0).reshape(G, N)          # (164, 201) total household expenditure
print(f'    fp_income total: {fp_income.sum():.4e} PDF')
print(f'    expenditure total: {expenditure.sum():.4e}')

# ── Step 4: sector footprint attributed to the final purchased product ─────
print('[4/4] Attributing footprint to final purchased product ...')
f_mat = f_vec.reshape(G, S)
fp_sector = np.zeros((S, G, N), dtype=np.float64)
for j in range(G):
    target_j = Target[:, j * N:(j + 1) * N].reshape(G, S, N)
    fp_sector[:, j, :] = np.einsum('cs,csn->sn', f_mat, target_j)
print(f'    fp_sector total: {fp_sector.sum():.4e} PDF  (should match fp_income total)')

# ── Save raw matrices ────────────────────────────────────────────────────
np.save(os.path.join(OUT_DIR, 'footprint_income_2023.npy'), fp_income)
np.save(os.path.join(OUT_DIR, 'footprint_by_sector_2023.npy'), fp_sector)
np.save(os.path.join(OUT_DIR, 'expenditure_2023.npy'), expenditure)
with open(os.path.join(OUT_DIR, 'country_order_2023.txt'), 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(regnam))
print(f'\nDone in {(time.time() - t0) / 60:.1f} min. Raw matrices saved to {OUT_DIR}')


[1/4] Building footprint intensity q ...
    Total production-side footprint (2023): 7.6653e-02 PDF
[2/4] Building Leontief inverse L ... (5-20 min)
    L done  [104s]
[3/4] Applying household expenditure (Target) ...
    fp_income total: 6.5679e-02 PDF
    expenditure total: 6.3850e+10
[4/4] Attributing footprint to final purchased product ...
    fp_sector total: 6.5679e-02 PDF  (should match fp_income total)

Done in 2.3 min. Raw matrices saved to E:\phdstudy\BiodiversityPHD-2ndpaper\submission_code\1_Biodiversity_footprint_accounting\output


## Part 2 ? Aggregate to country / region / expenditure group / decile / sector

Uses the two raw matrices from Part 1 plus population data to compute total
and per-capita footprint at each aggregation level. The expenditure outputs
preserve the 201 household expenditure bins and additionally aggregate them to
10 population-weighted global expenditure deciles.


In [2]:
import numpy as np
import pandas as pd
import os

# -- Load raw matrices (rerun Part 1 first, or point to its OUT_DIR) --------
fp_income = np.load(os.path.join(OUT_DIR, 'footprint_income_2023.npy'))   # (164, 201)
fp_sector = np.load(os.path.join(OUT_DIR, 'footprint_by_sector_2023.npy'))  # (120, 164, 201)
exp_path = os.path.join(OUT_DIR, 'expenditure_2023.npy')
if os.path.exists(exp_path):
    expenditure = np.load(exp_path)                                      # (164, 201)
else:
    # Backward-compatible fallback for old raw outputs; Part 1 now saves this file.
    npz = np.load(os.path.join(HOU_DIR, 'Target_2023.npz'))
    expenditure = np.nan_to_num(npz['Target'], nan=0.0).sum(axis=0).reshape(G, N)
with open(os.path.join(OUT_DIR, 'country_order_2023.txt'), encoding='utf-8') as fh:
    regnam = fh.read().splitlines()

REF_DIR = r'E:\phdstudy\reference'
mapping = pd.read_csv(os.path.join(HOU_DIR, 'GLORIA_Country_Mapping.csv'))
pop_raw = pd.read_csv(os.path.join(HOU_DIR, 'Population_by_IncomeGroup.csv'), index_col=0)

# -- Population by country x expenditure bin (164, 201), aligned with fp_income --
pop = np.zeros((G, N))
for gi in range(G):
    for _, row in mapping[mapping['GLORIA_Index'] == gi + 1].iterrows():
        iso3 = row['Population_ISO3']
        if pd.notna(iso3) and iso3 != 'Not in Population' and iso3 in pop_raw.columns:
            pop[gi] += pop_raw[iso3].values

# -- Country level: total + per-capita -------------------------------------
country_total = fp_income.sum(axis=1)                              # (164,)
country_pop = pop.sum(axis=1)                                      # (164,)
country_percap = np.divide(country_total, country_pop,
                           out=np.full_like(country_total, np.nan, dtype=float),
                           where=country_pop > 0)
df_country = pd.DataFrame({
    'Country': regnam, 'Footprint_PDF': country_total,
    'Population': country_pop, 'PerCapita_PDF': country_percap,
})

# -- Region level: World Bank region classification ------------------------
# Use the same GLORIA-order WBR mapping as the N-series notebooks.
nat = (pd.read_csv(os.path.join(REF_DIR, 'nationlist_categorized_titlecase.csv'))
       .sort_values('Country_ID').reset_index(drop=True))
WBR_NAMES = {
    'EAP': 'East Asia & Pacific',
    'ECA': 'Europe & Central Asia',
    'LAC': 'Latin America & Caribbean',
    'MENA': 'Middle East & North Africa',
    'NAM': 'North America',
    'SAR': 'South Asia',
    'SSA': 'Sub-Saharan Africa',
}
region_codes = nat['WBR'].tolist()
assert len(region_codes) == G, f'WBR mapping length mismatch: {len(region_codes)} != {G}'
region_of_country = [WBR_NAMES.get(code, code) for code in region_codes]


def aggregate_countries_by(group_labels, order):
    """Sum footprint/population over countries sharing the same group label."""
    rows = []
    for label in order:
        idx = [j for j, g in enumerate(group_labels) if g == label]
        total = country_total[idx].sum() if idx else np.nan
        pop_sum = country_pop[idx].sum() if idx else np.nan
        rows.append({'Group': label, 'Footprint_PDF': total, 'Population': pop_sum,
                     'PerCapita_PDF': total / pop_sum if pop_sum else np.nan})
    return pd.DataFrame(rows)


REGION_ORDER = ['East Asia & Pacific', 'Europe & Central Asia',
                'Latin America & Caribbean', 'Middle East & North Africa',
                'North America', 'South Asia', 'Sub-Saharan Africa']

df_region = aggregate_countries_by(region_of_country, REGION_ORDER)

# -- Expenditure-bin level: preserve all 201 bins --------------------------
bin_total = fp_income.sum(axis=0)                                  # (201,)
bin_pop = pop.sum(axis=0)                                          # (201,)
bin_percap = np.divide(bin_total, bin_pop,
                       out=np.full_like(bin_total, np.nan, dtype=float),
                       where=bin_pop > 0)
df_expenditure_group = pd.DataFrame({
    'BinIndex': np.arange(N),
    'ExpenditureGroup': [f'Bin{i}' for i in range(N)],
    'Footprint_PDF': bin_total,
    'Population': bin_pop,
    'PerCapita_PDF': bin_percap,
})

# -- Expenditure deciles: population-weighted global deciles from 201 bins --
def aggregate_to_global_expenditure_deciles(footprint, population, expenditure, n_deciles=10):
    """Aggregate country-bin cells into N7-style global expenditure deciles.

    Cells are ranked by per-capita expenditure (expenditure / population).
    Boundary cells are split proportionally by population so decile totals
    conserve footprint, expenditure, and population.
    """
    fp = np.asarray(footprint, dtype=float).reshape(-1)
    pop_arr = np.asarray(population, dtype=float).reshape(-1)
    exp_arr = np.asarray(expenditure, dtype=float).reshape(-1)

    with np.errstate(divide='ignore', invalid='ignore'):
        exp_pc = np.where(pop_arr > 0, exp_arr / pop_arr, np.nan)

    valid = (pop_arr > 0) & np.isfinite(exp_pc)
    fp_v = np.nan_to_num(fp[valid], nan=0.0)
    pop_v = pop_arr[valid]
    exp_v = np.nan_to_num(exp_arr[valid], nan=0.0)
    exp_pc_v = exp_pc[valid]

    order = np.argsort(exp_pc_v, kind='mergesort')
    fp_s = fp_v[order]
    pop_s = pop_v[order]
    exp_s = exp_v[order]

    total_pop = float(pop_s.sum())
    if total_pop <= 0:
        return pd.DataFrame({
            'Decile': [f'D{i}' for i in range(1, n_deciles + 1)],
            'Footprint_PDF': np.nan,
            'Population': np.nan,
            'Expenditure': np.nan,
            'PerCapita_PDF': np.nan,
        })

    target_pop = total_pop / n_deciles
    dec_fp = np.zeros(n_deciles, dtype=float)
    dec_pop = np.zeros(n_deciles, dtype=float)
    dec_exp = np.zeros(n_deciles, dtype=float)

    decile = 0
    remaining_target = target_pop
    for fp_i, pop_i, exp_i in zip(fp_s, pop_s, exp_s):
        pop_left = float(pop_i)
        while pop_left > 0 and decile < n_deciles:
            take = min(pop_left, remaining_target)
            frac = take / pop_i
            dec_fp[decile] += fp_i * frac
            dec_exp[decile] += exp_i * frac
            dec_pop[decile] += take
            pop_left -= take
            remaining_target -= take
            if remaining_target <= target_pop * 1e-10:
                decile += 1
                remaining_target = target_pop
        if decile >= n_deciles:
            break

    return pd.DataFrame({
        'Decile': [f'D{i}' for i in range(1, n_deciles + 1)],
        'Footprint_PDF': dec_fp,
        'Population': dec_pop,
        'Expenditure': dec_exp,
        'PerCapita_PDF': np.divide(dec_fp, dec_pop,
                                   out=np.full(n_deciles, np.nan, dtype=float),
                                   where=dec_pop > 0),
    })

df_decile = aggregate_to_global_expenditure_deciles(fp_income, pop, expenditure)

# -- Sector level: global total + global per-capita ------------------------
sector_total = fp_sector.sum(axis=(1, 2))                          # (120,)
global_pop = country_pop.sum()
df_sector = pd.DataFrame({
    'Sector_index': np.arange(S), 'Footprint_PDF': sector_total,
    'Share_pct': sector_total / sector_total.sum() * 100,
    'PerCapita_PDF': sector_total / global_pop,
})

# -- Export ----------------------------------------------------------------
out_xlsx = os.path.join(OUT_DIR, 'footprint_summary_2023.xlsx')
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as w:
    df_country.to_excel(w, sheet_name='Country', index=False)
    df_region.to_excel(w, sheet_name='Region', index=False)
    df_expenditure_group.to_excel(w, sheet_name='ExpenditureGroup', index=False)
    df_decile.to_excel(w, sheet_name='Decile', index=False)
    df_sector.to_excel(w, sheet_name='Sector', index=False)
print(f'Saved aggregated summary: {out_xlsx}')
print(f'  ExpenditureGroup rows: {len(df_expenditure_group)}')
print(f'  Decile rows: {len(df_decile)}')


Saved aggregated summary: E:\phdstudy\BiodiversityPHD-2ndpaper\submission_code\1_Biodiversity_footprint_accounting\output\footprint_summary_2023.xlsx
  ExpenditureGroup rows: 201
  Decile rows: 10
